# 6단계: 종합 인사이트 리포트
## 모든 분석을 하나로 — 투자 판단 자동화

---
### 이 실습에서 배우는 것
- 함수로 분석 모듈화
- 조건문으로 자동 판단 시스템
- 서브플롯으로 대시보드 구성
- 결과를 텍스트 리포트로 출력

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import platform
from datetime import datetime

if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

# 전체 데이터 로드
df = pd.read_csv('data/coway_annual.csv')
df_q = pd.read_csv('data/coway_quarterly.csv')
df_os = pd.read_csv('data/coway_overseas.csv')

for col in ['매출액', '영업이익', '법인세전이익', '당기순이익', '국내매출', '해외매출', '기타매출']:
    df[col] = df[col] / 100
for col in ['매출액', '영업이익', '법인세전이익', '당기순이익']:
    df_q[col] = df_q[col] / 100

# 지표 사전 계산
df['OPM'] = (df['영업이익'] / df['매출액'] * 100).round(1)
df['NPM'] = (df['당기순이익'] / df['매출액'] * 100).round(1)
df['매출_YoY'] = df['매출액'].pct_change() * 100
df['영업이익_YoY'] = df['영업이익'].pct_change() * 100
df['계정순증_천'] = df['렌탈계정수_천'].diff()
df['해외비중'] = (df['해외매출'] / df['매출액'] * 100).round(1)
df['EPS'] = (df['당기순이익'] * 1e8) / df['주식수']
df_q['OPM'] = (df_q['영업이익'] / df_q['매출액'] * 100).round(1)

# 현재 주가 (실제 분석 시 업데이트)
현재주가 = 62000

latest = df[df['연도'] == 2025].iloc[0]
print('데이터 준비 완료')

## 6-1. 투자 체크리스트 자동화

In [ ]:
def investment_checklist(df, df_os, 현재주가):
    """
    코웨이 투자 체크리스트 자동 평가
    Returns: dict {항목: (통과여부, 상세내용)}
    """
    latest = df[df['연도'] == df['연도'].max()].iloc[0]
    prev = df[df['연도'] == df['연도'].max() - 1].iloc[0]
    
    결과 = {}
    
    # 1. 렌탈 계정수 순증
    계정순증 = latest['렌탈계정수_천'] - prev['렌탈계정수_천']
    결과['렌탈 계정수 성장'] = (
        계정순증 > 0,
        f"+{계정순증:.0f}천 계정 순증" if 계정순증 > 0 else f"{계정순증:.0f}천 계정 감소"
    )
    
    # 2. OPM 18% 이상
    opm = latest['OPM']
    결과['영업이익률 18% 이상'] = (
        opm >= 18,
        f"현재 OPM {opm}%"
    )
    
    # 3. 매출 성장
    매출성장 = latest['매출_YoY']
    결과['매출 YoY 성장'] = (
        매출성장 > 0,
        f"YoY +{매출성장:.1f}%" if 매출성장 > 0 else f"YoY {매출성장:.1f}%"
    )
    
    # 4. 해외 매출 비중 확대
    해외비중 = latest['해외비중']
    prev_해외비중 = prev['해외비중']
    결과['해외 비중 확대'] = (
        해외비중 > prev_해외비중,
        f"{prev_해외비중}% → {해외비중}% (변화: {해외비중 - prev_해외비중:+.1f}%p)"
    )
    
    # 5. PER 적정
    EPS = latest['EPS']
    PER = 현재주가 / EPS
    결과['PER 15배 이하'] = (
        PER <= 15,
        f"현재 PER {PER:.1f}배"
    )
    
    # 6. 배당수익률
    DPS = 3700
    배당수익률 = DPS / 현재주가 * 100
    결과['배당수익률 3% 이상'] = (
        배당수익률 >= 3.0,
        f"배당수익률 {배당수익률:.1f}%"
    )
    
    # 7. 미국 법인 흑자전환
    us_profit = df_os[(df_os['법인'] == '미국') & (df_os['연도'] == df['연도'].max())]['영업이익_억원'].values
    us_흑자 = len(us_profit) > 0 and us_profit[0] > 0
    결과['미국 법인 흑자'] = (
        us_흑자,
        f"미국 법인 영업이익 {us_profit[0]:+.0f}억원" if len(us_profit) > 0 else "데이터 없음"
    )
    
    return 결과

# 체크리스트 실행
체크결과 = investment_checklist(df, df_os, 현재주가)

print(f'\n{'='*55}')
print(f'  코웨이 투자 체크리스트 ({datetime.now().strftime("%Y.%m.%d")})')
print(f'{'='*55}')

통과 = 0
전체 = len(체크결과)

for 항목, (결과, 상세) in 체크결과.items():
    아이콘 = '✅' if 결과 else '❌'
    통과 += int(결과)
    print(f'{아이콘} {항목:<20} | {상세}')

print(f'{'='*55}')
print(f'총점: {통과}/{전체}점')

if 통과 >= 6:
    print('\n🟢 종합 판단: 매수 고려 가능')
elif 통과 >= 4:
    print('\n🟡 종합 판단: 중립 — 추가 모니터링 필요')
else:
    print('\n🔴 종합 판단: 투자 신중 — 리스크 재점검')

## 6-2. 종합 대시보드

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# ① 매출 & 영업이익 트렌드
ax1 = fig.add_subplot(gs[0, :])
ax1_r = ax1.twinx()
ax1.bar(df['연도'] - 0.2, df['매출액'], width=0.35, color='steelblue', alpha=0.7, label='매출액')
ax1.bar(df['연도'] + 0.2, df['영업이익'], width=0.35, color='orange', alpha=0.8, label='영업이익')
ax1_r.plot(df['연도'], df['OPM'], 'ro-', linewidth=2, markersize=7, label='OPM')
ax1_r.axhline(y=18, color='red', linestyle='--', alpha=0.4)
ax1_r.set_ylabel('OPM (%)', color='red')
ax1_r.tick_params(axis='y', labelcolor='red')
ax1.set_title('① 연간 실적 & 영업이익률', fontweight='bold')
ax1.set_ylabel('금액 (억원)')
lines1, lbl1 = ax1.get_legend_handles_labels()
lines2, lbl2 = ax1_r.get_legend_handles_labels()
ax1.legend(lines1 + lines2, lbl1 + lbl2, loc='upper left', fontsize=8)

# ② 렌탈 계정수 추이
ax2 = fig.add_subplot(gs[1, 0])
ax2.bar(df['연도'], df['렌탈계정수_천'] / 100, color='#4CAF50', alpha=0.7)
ax2.set_title('② 렌탈 계정수 (만)', fontweight='bold')
ax2.set_ylabel('만 계정')
for y, v in zip(df['연도'], df['렌탈계정수_천'] / 100):
    ax2.text(y, v + 0.5, f'{v:.0f}만', ha='center', fontsize=8)

# ③ 사업부문 구성 (최신)
ax3 = fig.add_subplot(gs[1, 1])
latest_pie = df[df['연도'] == 2025].iloc[0]
ax3.pie([latest_pie['국내매출'], latest_pie['해외매출'], latest_pie['기타매출']],
       labels=['국내', '해외', '기타'],
       colors=['#2196F3', '#FF9800', '#9E9E9E'],
       autopct='%1.0f%%', startangle=90)
ax3.set_title('③ 2025 사업부문 비중', fontweight='bold')

# ④ 해외법인 매출 비교
ax4 = fig.add_subplot(gs[1, 2])
for 법인, color in [('말레이시아', '#FF9800'), ('미국', '#2196F3'), ('태국', '#4CAF50')]:
    d = df_os[df_os['법인'] == 법인]
    ax4.plot(d['연도'], d['매출액_억원'], 'o-', color=color, label=법인, linewidth=2)
ax4.axhline(y=0, color='black', linewidth=0.5)
ax4.set_title('④ 해외법인 매출 추이', fontweight='bold')
ax4.set_ylabel('매출액 (억원)')
ax4.legend(fontsize=8)
ax4.grid(True, alpha=0.3)

# ⑤ 분기별 OPM 히트맵
ax5 = fig.add_subplot(gs[2, 0:2])
df_q['연도'] = df_q['연도'].astype(str)
pivot_opm = df_q.pivot(index='분기번호', columns='연도', values='OPM')
im = ax5.imshow(pivot_opm.values, cmap='RdYlGn', aspect='auto', vmin=10, vmax=22)
ax5.set_xticks(range(len(pivot_opm.columns)))
ax5.set_xticklabels(pivot_opm.columns)
ax5.set_yticks(range(4))
ax5.set_yticklabels(['Q1', 'Q2', 'Q3', 'Q4'])
for i in range(4):
    for j in range(len(pivot_opm.columns)):
        val = pivot_opm.values[i, j]
        ax5.text(j, i, f'{val:.1f}%', ha='center', va='center', fontsize=9, fontweight='bold')
plt.colorbar(im, ax=ax5, shrink=0.8)
ax5.set_title('⑤ 분기별 OPM 히트맵 (초록=높음, 빨강=낮음)', fontweight='bold')

# ⑥ 투자 체크리스트 스코어보드
ax6 = fig.add_subplot(gs[2, 2])
항목들 = list(체크결과.keys())
점수들 = [1 if v[0] else 0 for v in 체크결과.values()]
colors_bar = ['#4CAF50' if s else '#F44336' for s in 점수들]
y_pos = range(len(항목들))
ax6.barh(y_pos, 점수들, color=colors_bar, alpha=0.8)
ax6.set_yticks(y_pos)
ax6.set_yticklabels([h[:10] for h in 항목들], fontsize=8)
ax6.set_xlim(0, 1.3)
ax6.set_title(f'⑥ 투자 체크리스트\n({sum(점수들)}/{len(점수들)}점)', fontweight='bold')
ax6.set_xticks([])
for y, s, 항목 in zip(y_pos, 점수들, 항목들):
    ax6.text(1.05, y, '✅' if s else '❌', va='center', fontsize=12)

fig.suptitle('코웨이(021240) 종합 재무 분석 대시보드',
            fontsize=15, fontweight='bold', y=1.01)

plt.savefig('data/dashboard_final.png', dpi=150, bbox_inches='tight')
plt.show()
print('대시보드 저장 완료: data/dashboard_final.png')

## 6-3. 텍스트 인사이트 리포트 자동 생성

In [ ]:
def generate_report(df, df_os, 체크결과, 현재주가):
    latest = df[df['연도'] == df['연도'].max()].iloc[0]
    prev = df[df['연도'] == df['연도'].max() - 1].iloc[0]
    
    def cagr(s, e, n): return ((e/s)**(1/n)-1)*100
    매출_cagr = cagr(df[df['연도']==2022]['매출액'].values[0], latest['매출액'], 3)
    
    EPS = latest['EPS']
    PER = 현재주가 / EPS
    
    report = f"""
{'='*60}
  코웨이 재무분석 리포트 ({datetime.now().strftime('%Y년 %m월 %d일')})
{'='*60}

[1. 성장성 요약]
  - 2025 매출액: {latest['매출액']:,.0f}억원 (YoY +{latest['매출_YoY']:.1f}%)
  - 3년 매출 CAGR (2022→2025): {매출_cagr:.1f}%
  - 렌탈 계정수: {latest['렌탈계정수_천']/100:.0f}만 계정 (전년比 +{latest['계정순증_천']:.0f}천)

[2. 수익성 요약]
  - 영업이익률(OPM): {latest['OPM']}% {'✅ 18% 이상 유지' if latest['OPM'] >= 18 else '⚠️ 18% 미만'}
  - 순이익률(NPM): {latest['NPM']}%
  - 영업이익 YoY: {latest['영업이익_YoY']:+.1f}%

[3. 해외사업 요약]
  - 해외 매출 비중: {latest['해외비중']}% (전년 {prev['해외비중']}%)
  - 말레이시아: {df_os[(df_os['법인']=='말레이시아')&(df_os['연도']==2025)]['매출액_억원'].values[0]:,.0f}억원
  - 미국: {df_os[(df_os['법인']=='미국')&(df_os['연도']==2025)]['매출액_억원'].values[0]:,.0f}억원
    {'(흑자전환 ✅)' if df_os[(df_os['법인']=='미국')&(df_os['연도']==2025)]['영업이익_억원'].values[0] > 0 else '(적자 지속 ⚠️)'}
  - 태국: {df_os[(df_os['법인']=='태국')&(df_os['연도']==2025)]['매출액_억원'].values[0]:,.0f}억원

[4. 밸류에이션]
  - 현재 주가: {현재주가:,}원
  - EPS: {EPS:,.0f}원 / PER: {PER:.1f}배
  - 적정주가(PER 13배): {EPS * 13:,.0f}원
  - 현재가 대비: {(EPS*13/현재주가 - 1)*100:+.1f}%

[5. 투자 판단]
  체크리스트 점수: {sum(v[0] for v in 체크결과.values())}/{len(체크결과)}점
  종합 의견: {'매수 고려' if sum(v[0] for v in 체크결과.values()) >= 6 else '중립 — 추가 모니터링' if sum(v[0] for v in 체크결과.values()) >= 4 else '투자 신중'}

[6. 주요 리스크]
  - 4Q 영업이익 계절성 하락 패턴 반복 여부
  - 말레이시아 환율 영향 (링깃 약세 시 원화 환산 매출 감소)
  - 넷마블(대주주) 재무상황에 따른 지분 매각 리스크
  - 국내 시장 포화에 따른 계정 순증 둔화

{'='*60}
※ 본 분석은 학습 목적으로 작성되었으며 투자 권유가 아닙니다.
{'='*60}
"""
    return report

리포트 = generate_report(df, df_os, 체크결과, 현재주가)
print(리포트)

# 파일로도 저장
with open('data/coway_report.txt', 'w', encoding='utf-8') as f:
    f.write(리포트)
print('리포트 저장 완료: data/coway_report.txt')

---
## ✏️ 최종 과제 (종합)

1. `investment_checklist()` 함수에 항목을 하나 더 추가해보세요.
   - 예: `'ARPU 증가 여부'`, `'해외 비중 35% 이상'` 등

2. `generate_report()` 함수를 수정해서 4분기 계절성 분석 결과도 포함시켜보세요.

3. 다른 기업(코웨이 경쟁사 등)의 데이터를 추가해서 `비교 분석` 차트를 만들어보세요.
   - 힌트: `df_경쟁사`를 만들어서 같은 차트에 다른 색으로 표시

4. 이 분석에서 가장 중요하다고 생각하는 지표 3개와 이유를 한 문단으로 작성해보세요.